In [ ]:
# ===========================================
# Automação Zendesk — N1
# Solicitar Pedido de Compra + Pagamento NF/CTe
# Baseado no N2, adaptado para o fluxo N1
# ===========================================

# ---- Dependências ----
import sys, subprocess, importlib.util
def ensure_installed(*pkgs):
    for p in pkgs:
        if importlib.util.find_spec(p) is None:
            print(f"Instalando {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])
    print("✓ Dependências checadas.")

ensure_installed("pandas", "xlwings", "selenium", "python-dateutil", "openpyxl")

# ---- Imports principais ----
import os
import time
import re
import getpass
import pandas as pd
import xlwings as xw
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import (
    TimeoutException, WebDriverException,
    NoSuchElementException, ElementNotInteractableException
)
from datetime import datetime, timedelta
from dateutil import parser

# ================================================================
# CONFIGURAÇÕES GERAIS
# ================================================================
LOGIN_FIXO   = "126815@pmenos.com.br"
EXCEL_PATH   = r"C:\Users\126815\OneDrive - paguemenos.com.br\ENGENHARIA - OBRAS - Documentos\BANCO DE DADOS - POWER BI\BI - OUTROS\BASE CONTROLE DE PAGAMENTOS.xlsx"
FORM_URL     = "https://portaldeservicos.pmenos.com.br/hc/pt-br/requests/new?ticket_form_id=41684359104269"
BASE_PDF_DIR = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

# Campos FIXOS do formulário N1
SETOR_APROVACAO  = "Obras"
LOCAL_ATENDIMENTO = "Matriz"
MATERIAL_VALOR   = "Material Ativo"
COD_MATERIAL     = "20272"
BANDEIRA_LIMITE  = 7000   # < 7000 → PM | >= 7000 → EF

# Perfil Chrome persistente
CHROME_PROFILE_PATH = os.path.join(os.getcwd(), "ChromeProfile")
os.makedirs(CHROME_PROFILE_PATH, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(
        "❌ Planilha não encontrada:\n"
        f"   {EXCEL_PATH}\n"
        "💡 Verifique se o OneDrive sincronizou ('Disponível neste dispositivo')."
    )

# ================================================================
# FUNÇÕES UTILITÁRIAS — localizar e preencher por label
# ================================================================
def _normalize_text(txt: str) -> str:
    return re.sub(r'\s+', ' ', (txt or '').strip()).lower()

def find_input_by_label(driver, label_candidates, timeout=10):
    """Localiza input/textarea/select por label, aria-label, name ou placeholder."""
    norm_targets = [_normalize_text(l) for l in label_candidates]

    # 1) label[for] → id
    try:
        labels = driver.find_elements(By.TAG_NAME, "label")
        for lbl in labels:
            try:
                lbl_text = _normalize_text(lbl.text)
                if any(lbl_text.startswith(t) for t in norm_targets):
                    for_attr = lbl.get_attribute("for")
                    if for_attr:
                        try:
                            return driver.find_element(By.ID, for_attr)
                        except NoSuchElementException:
                            pass
            except Exception:
                continue
    except Exception:
        pass

    # 2) aria-label / name / placeholder
    for cand in label_candidates:
        norm = _normalize_text(cand)
        for attr in ['aria-label', 'name', 'placeholder']:
            try:
                elems = driver.find_elements(By.CSS_SELECTOR, f'[{attr}]')
                for e in elems:
                    val = _normalize_text(e.get_attribute(attr))
                    if val.startswith(norm) or val == norm.replace(' (opcional)', ''):
                        return e
            except Exception:
                pass

    # 3) Fallback: campos custom Zendesk
    try:
        candidates = driver.find_elements(
            By.CSS_SELECTOR,
            'input[id^="request_custom_fields_"], textarea[id^="request_custom_fields_"], select[id^="request_custom_fields_"]'
        )
        for elem in candidates:
            try:
                if elem.is_displayed() and elem.is_enabled():
                    parent = elem.find_element(By.XPATH, './ancestor::div[1]')
                    maybe_label = parent.find_element(By.TAG_NAME, 'label')
                    text = _normalize_text(maybe_label.text)
                    if any(text.startswith(t) for t in norm_targets):
                        return elem
            except Exception:
                continue
    except Exception:
        pass

    return None

def fill_text_or_select(driver, element, value: str):
    """Preenche input/textarea ou select (incluindo select2-like)."""
    if not value:
        return
    tag = element.tag_name.lower()
    try:
        if tag == 'select':
            try:
                Select(element).select_by_visible_text(value)
            except Exception:
                try:
                    Select(element).select_by_value(value)
                except Exception:
                    element.click()
                    time.sleep(0.3)
                    opts = driver.find_elements(
                        By.XPATH,
                        f'//li[contains(@class,"select2-results__option") and normalize-space()="{value}"]'
                        f' | //div[contains(@class,"option") and normalize-space()="{value}"]'
                    )
                    if opts:
                        opts[0].click()
        elif tag in ('input', 'textarea'):
            try:
                element.clear()
            except Exception:
                pass
            element.send_keys(value)
        else:
            try:
                element.click()
                element.send_keys(value)
            except Exception:
                pass
    except (ElementNotInteractableException, NoSuchElementException):
        pass

def preencher_campo_por_label(driver, label_candidates: list, value: str, timeout=12):
    """
    Localiza um campo por label e preenche.
    Não falha caso o campo não exista no formulário.
    """
    value = (value or "").strip()
    if not value:
        return

    elem = find_input_by_label(driver, label_candidates, timeout=timeout)
    if elem is None:
        try:
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'input, textarea, select'))
            )
            elem = find_input_by_label(driver, label_candidates, timeout=5)
        except TimeoutException:
            elem = None

    if elem:
        fill_text_or_select(driver, elem, value)
    else:
        print(f"⚠️ Campo '{label_candidates[0]}' não encontrado. Seguindo sem preencher.")

# ================================================================
# FUNÇÕES DE LOGIN
# ================================================================
def inicializar_driver():
    chrome_options = Options()
    # Na PRIMEIRA execução, comente a linha abaixo para autenticar (MFA):
    # chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument(f"--user-data-dir={CHROME_PROFILE_PATH}")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    )
    chrome_options.add_experimental_option('excludeSwitches', ['enable-logging', 'enable-automation'])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": "C:/temp",
        "download.prompt_for_download": False,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_settings.popups": 0
    })

    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.set_page_load_timeout(45)
        driver.implicitly_wait(10)
        driver.get(FORM_URL)
        print("✅ Driver Chrome inicializado com sucesso")
        return driver
    except Exception as e:
        print(f"❌ Erro ao inicializar Chrome: {e}")
        raise

def realizar_login(driver, login_fixo):
    print("\n" + "="*64)
    print("                 VERIFICANDO ESTADO DE LOGIN")
    print("="*64)

    try:
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "request_subject")))
        print("✅ Sessão já ativa. Login ignorado.")
        return driver
    except:
        print("ℹ️ Sessão não encontrada. Iniciando login...")

    senha = getpass.getpass(f"🔑 Digite a senha para {login_fixo}: ")

    # 1. Login
    try:
        seletores_login = [
            (By.ID, "identifierId"), (By.NAME, "loginfmt"),
            (By.ID, "i0116"), (By.CSS_SELECTOR, "input[type='email']")
        ]
        campo_login = None
        for by, sel in seletores_login:
            try:
                campo_login = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((by, sel)))
                if campo_login: break
            except: continue
        if campo_login:
            campo_login.clear()
            campo_login.send_keys(login_fixo)
            campo_login.send_keys(Keys.ENTER)
            print("✅ Login inserido.")
            time.sleep(2)
    except Exception as e:
        print(f"❌ Erro no login: {str(e)[:100]}")

    # 2. Senha
    try:
        seletores_senha = [
            (By.NAME, "password"), (By.NAME, "passwd"),
            (By.ID, "i0118"), (By.CSS_SELECTOR, "input[type='password']")
        ]
        campo_senha = None
        for by, sel in seletores_senha:
            try:
                campo_senha = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, sel)))
                if campo_senha: break
            except: continue
        if campo_senha:
            campo_senha.send_keys(senha)
            time.sleep(1)
            campo_senha.send_keys(Keys.ENTER)
            print("✅ Senha inserida.")
            time.sleep(3)
    except Exception as e:
        print(f"❌ Erro na senha: {str(e)[:100]}")

    # 3. MFA / Token
    try:
        seletores_token = [
            (By.ID, "idTxtBx_SAOTCC_OTC"), (By.NAME, "otc"),
            (By.CSS_SELECTOR, "input[aria-label='Código de verificação']"),
            (By.ID, "idTxtBx_SAOTAS_OTC"),
        ]
        campo_token = None
        start = time.time()
        while time.time() - start < 20:
            for by, sel in seletores_token:
                try:
                    campo_token = driver.find_element(by, sel)
                    if campo_token.is_displayed(): break
                except: continue
            if campo_token: break
            try:
                msg_app = driver.find_element(By.ID, "idDiv_SAOTCAS_Title")
                if "Aprovar" in msg_app.text or "App" in msg_app.text:
                    print("📱 Aprovação via App detectada. Aprove no celular...")
                    break
            except: pass
            time.sleep(1)

        if campo_token:
            token = input("🔒 Digite o Token MFA e pressione Enter: ")
            campo_token.send_keys(token)
            time.sleep(1)
            campo_token.send_keys(Keys.ENTER)
            print("✅ Token inserido.")
    except Exception as e:
        print(f"❌ Erro no token: {str(e)[:100]}")

    # 4. Continuar conectado?
    try:
        seletores_sim = [
            (By.ID, "idSIButton9"),
            (By.CSS_SELECTOR, "input[type='submit'][value='Sim']"),
            (By.XPATH, "//input[@value='Sim']")
        ]
        btn_sim = None
        start = time.time()
        while time.time() - start < 15:
            for by, sel in seletores_sim:
                try:
                    btn_sim = driver.find_element(by, sel)
                    if btn_sim.is_displayed() and btn_sim.is_enabled(): break
                except: continue
            if btn_sim: break
            time.sleep(1)
        if btn_sim:
            try: btn_sim.click()
            except: driver.execute_script("arguments[0].click();", btn_sim)
            print("✅ 'Continuar conectado' confirmado.")
    except: pass

    print("="*64)
    print("             LOGIN CONCLUÍDO")
    print("="*64)
    time.sleep(5)
    return driver

# ================================================================
# LÓGICA DE ABERTURA DE CHAMADO N1
# ================================================================
def abrir_chamado_n1(row_index, row_data, driver, wb, ws):
    """
    Preenche o formulário N1 no Zendesk.

    Campos do formulário (ordem visual):
      1. Serviços financeiros ao fornecedor  → Regularização de Notas Fiscais   (Tab→4↓)
      2. Regularização de Notas Fiscais      → Lançamento Reg. NF de Serviço    (Tab→1↓)
      3. Setor de Aprovação                  → Obras (fixo, por label)
      4. Centro de Custo                     → FILIAL da planilha (por label)
      5. Forma de Pagamento                  → Crédito em Conta                  (Tab→2↓)
      6. Ordem                               → ORDEM da planilha (Tab+texto)
      7. Valor                               → VALOR BI da planilha (Tab+texto)
      8. Data de Vencimento                  → extraída do ASSUNTO ou +7 dias    (Tab+texto)
      9. Cod. Fornecedor                     → por label (opcional)
     10. Bandeira                            → PM ou EF conforme filial (por label)
     11. Fornecedor                          → por label (opcional)
     12. Material                            → Material Ativo (por label)
     13. Cód Material                        → 20272 (por label)
     14. Local de Atendimento                → Matriz (Tab→select)
     15. Assunto                             → campo #request_subject
     16. Descrição                           → campo #request_description
     17. Checkbox [Anexo] Nota Fiscal        → Tab+Espaço
     18. Anexo PDF                           → send_keys no input file
    """

    # ── Leitura dos dados da linha ──────────────────────────────
    assunto = str(row_data["ASSUNTO"]).strip()

    # Ordem de compra
    ordem_raw = row_data.get("ORDEM", "")
    if pd.notna(ordem_raw) and str(ordem_raw).replace('.', '', 1).isdigit():
        ordem = str(int(float(ordem_raw))).strip()
    else:
        ordem = str(ordem_raw).strip() if pd.notna(ordem_raw) else ""

    # Filial → Centro de Custo
    filial_raw = row_data.get("FILIAL", "")
    filial = str(int(float(filial_raw))).strip() if pd.notna(filial_raw) and str(filial_raw).replace('.','',1).isdigit() else str(filial_raw).strip()

    # Bandeira: PM se filial < 7000, EF se >= 7000
    try:
        bandeira = "PM" if int(filial) < BANDEIRA_LIMITE else "EF"
    except Exception:
        bandeira = "PM"

    # Valor
    valor_raw = str(row_data.get("VALOR BI", "0")).strip()
    try:
        valor_limpo = re.sub(r"[^\d.,]", "", valor_raw)
        if re.match(r"^\d{1,3}(\.\d{3})*,\d{2}$", valor_limpo):
            valor_formatado = valor_limpo
        else:
            valor_num = float(valor_limpo.replace(',', '.'))
            valor_formatado = f"{valor_num:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        valor_formatado = "0,00"

    # Data de Vencimento
    venc_match = re.search(r"VENC\s*(\d{2}/\d{2}/\d{4})", assunto)
    if venc_match:
        try:
            data_obj = parser.parse(venc_match.group(1), dayfirst=True)
            data_vencimento = data_obj.strftime("%d/%m/%Y")
        except Exception:
            data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")
    else:
        data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")

    # Extração de campos do assunto
    cd_match = re.search(r"CD\s*([A-Z0-9]{2,4})", assunto, re.IGNORECASE)
    if cd_match:
        loja = f"CD {cd_match.group(1).strip()}"
        projeto_match = re.match(r"(.+?)\s*CD\s*[A-Z0-9]{2,4}", assunto, re.IGNORECASE)
        projeto = projeto_match.group(1).strip() if projeto_match else ""
    else:
        proj_loja = re.match(r"([A-Z\s]+)\s+(\d{3,4})", assunto)
        projeto = proj_loja.group(1).strip() if proj_loja else ""
        loja    = proj_loja.group(2).strip() if proj_loja else ""

    if not projeto:
        fb = re.match(r"(.+?)(?=\s*-\s*|\s*N[°º])", assunto, re.IGNORECASE)
        projeto = fb.group(1).strip() if fb else ""

    fornecedor_match = re.search(r"-\s*([A-ZÀ-Úa-z0-9 ]+?)(?=\s*-|\s*N[°º])", assunto)
    fornecedor = fornecedor_match.group(1).strip() if fornecedor_match else ""

    nf_match = re.search(r"N[°º]?\s*(\d+)", assunto)
    nf = nf_match.group(1) if nf_match else ""

    # Cod. Fornecedor (coluna COD_FORNECEDOR se existir)
    cod_fornecedor = ""
    try:
        if "COD_FORNECEDOR" in row_data.index and pd.notna(row_data["COD_FORNECEDOR"]):
            cod_fornecedor = str(row_data["COD_FORNECEDOR"]).strip()
    except Exception:
        pass

    # Fornecedor da base tem prioridade
    fornecedor_base = ""
    try:
        if "FORNECEDOR" in row_data.index and pd.notna(row_data["FORNECEDOR"]):
            fornecedor_base = str(row_data["FORNECEDOR"]).strip()
    except Exception:
        pass
    fornecedor_final = fornecedor_base or fornecedor

    # Descrição
    descricao_final = f"{assunto} | ORDEM: {ordem}"

    # PDF
    if loja.startswith("CD "):
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - {loja}.pdf")
    else:
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - LJ {loja}.pdf")
    pdf_existe = os.path.exists(pdf_path)

    # ── Log ─────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"🟩 N1 — PROCESSANDO LINHA {row_index + 2}")
    print(f"{'='*60}")
    print(f"📄 NF: {nf} | 🏪 Loja: {loja} | 🏗️ Projeto: {projeto}")
    print(f"🏢 Fornecedor: {fornecedor_final} | 💰 Valor: {valor_formatado}")
    print(f"🏦 Filial: {filial} | 🚩 Bandeira: {bandeira} | 📋 Ordem: {ordem}")
    print(f"📅 Vencimento: {data_vencimento}")
    print(f"📁 PDF: {'✅ Encontrado' if pdf_existe else '❌ Não encontrado'}")
    print(f"{'='*60}\n")

    try:
        time.sleep(3)
        actions = ActionChains(driver)

        # ── 1. DROPDOWN — Serviços financeiros ao fornecedor ────
        # Tab → Espaço → 4↓ → Enter  (Regularização de Notas Fiscais)
        print("⏳ Passo 1: Serviços financeiros → Regularização de Notas Fiscais (4↓)...")
        actions.send_keys(Keys.TAB).perform(); time.sleep(0.5)
        actions.send_keys(Keys.SPACE).perform(); time.sleep(0.5)
        for _ in range(4):
            actions.send_keys(Keys.ARROW_DOWN).perform(); time.sleep(0.1)
        actions.send_keys(Keys.ENTER).perform(); time.sleep(1)

        # ── 2. DROPDOWN — Regularização de Notas Fiscais ────────
        # Tab → Espaço → 1↓ → Enter  (Lançamento Regularização de NF de Serviço)
        print("⏳ Passo 2: Regularização NF → Lançamento Reg. NF de Serviço (1↓)...")
        actions.send_keys(Keys.TAB).perform(); time.sleep(0.3)
        actions.send_keys(Keys.SPACE).perform(); time.sleep(0.5)
        actions.send_keys(Keys.ARROW_DOWN).perform(); time.sleep(0.1)
        actions.send_keys(Keys.ENTER).perform(); time.sleep(1)

        # ── 3. SETOR DE APROVAÇÃO — fixo "Obras" ────────────────
        print("⏳ Passo 3: Setor de Aprovação → Obras...")
        preencher_campo_por_label(driver, ["Setor de Aprovação", "Setor de Aprovacao"], SETOR_APROVACAO)

        # ── 4. CENTRO DE CUSTO — número da filial ───────────────
        print(f"⏳ Passo 4: Centro de Custo → {filial}...")
        preencher_campo_por_label(driver, ["Centro de Custo"], filial)

        # ── 5. FORMA DE PAGAMENTO — Tab → Espaço → 2↓ → Enter ──
        # (Crédito em Conta)
        print("⏳ Passo 5: Forma de Pagamento → Crédito em Conta (2↓)...")
        actions.send_keys(Keys.TAB).perform(); time.sleep(0.3)
        actions.send_keys(Keys.SPACE).perform(); time.sleep(0.5)
        for _ in range(2):
            actions.send_keys(Keys.ARROW_DOWN).perform(); time.sleep(0.1)
        actions.send_keys(Keys.ENTER).perform(); time.sleep(0.5)

        # ── 6. ORDEM DE COMPRA ───────────────────────────────────
        print(f"⏳ Passo 6: Ordem → {ordem}...")
        actions.send_keys(Keys.TAB).send_keys(ordem).perform(); time.sleep(0.5)

        # ── 7. VALOR ─────────────────────────────────────────────
        print(f"⏳ Passo 7: Valor → {valor_formatado}...")
        actions.send_keys(Keys.TAB).send_keys(valor_formatado).perform(); time.sleep(0.5)

        # ── 8. DATA DE VENCIMENTO ────────────────────────────────
        print(f"⏳ Passo 8: Data de Vencimento → {data_vencimento}...")
        actions.send_keys(Keys.TAB).send_keys(data_vencimento).send_keys(Keys.ENTER).perform(); time.sleep(0.5)

        # ── 9. COD. FORNECEDOR (opcional) ───────────────────────
        if cod_fornecedor:
            print(f"⏳ Passo 9: Cod. Fornecedor → {cod_fornecedor}...")
            preencher_campo_por_label(driver, ["Cod.Fornecedor", "Cod. Fornecedor", "Código Fornecedor"], cod_fornecedor)

        # ── 10. BANDEIRA (PM / EF) ───────────────────────────────
        print(f"⏳ Passo 10: Bandeira → {bandeira}...")
        preencher_campo_por_label(driver, ["Bandeira"], bandeira)

        # ── 11. FORNECEDOR (opcional) ────────────────────────────
        print(f"⏳ Passo 11: Fornecedor → {fornecedor_final}...")
        preencher_campo_por_label(driver, ["Fornecedor", "Fornecedor (opcional)"], fornecedor_final)

        # ── 12. MATERIAL → Material Ativo ───────────────────────
        print("⏳ Passo 12: Material → Material Ativo...")
        preencher_campo_por_label(driver, ["Material", "Material (opcional)"], MATERIAL_VALOR)

        # ── 13. CÓD MATERIAL → 20272 ────────────────────────────
        print(f"⏳ Passo 13: Cód Material → {COD_MATERIAL}...")
        preencher_campo_por_label(driver, ["Cód Material", "Cod Material", "Código Material", "Cód. Material"], COD_MATERIAL)

        # ── 14. LOCAL DE ATENDIMENTO → Matriz ───────────────────
        # Tab → Select (select padrão)
        print("⏳ Passo 14: Local de Atendimento → Matriz...")
        actions.send_keys(Keys.TAB).perform(); time.sleep(0.3)
        try:
            local_elem = find_input_by_label(driver, ["Local de Atendimento"], timeout=8)
            if local_elem:
                fill_text_or_select(driver, local_elem, LOCAL_ATENDIMENTO)
            else:
                # fallback: tenta pelo Select visível mais próximo
                selects = driver.find_elements(By.TAG_NAME, "select")
                for s in selects:
                    if s.is_displayed():
                        try:
                            Select(s).select_by_visible_text(LOCAL_ATENDIMENTO)
                            break
                        except Exception:
                            continue
        except Exception as e:
            print(f"⚠️ Local de Atendimento: {e}")

        # ── 15. ASSUNTO ──────────────────────────────────────────
        print("⏳ Passo 15: Assunto...")
        campo_assunto = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "request_subject"))
        )
        campo_assunto.clear()
        campo_assunto.send_keys(assunto)

        # ── 16. DESCRIÇÃO ────────────────────────────────────────
        print("⏳ Passo 16: Descrição...")
        campo_descricao = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "request_description"))
        )
        campo_descricao.clear()
        campo_descricao.send_keys(descricao_final)

        # ── 17. CHECKBOX [Anexo] Nota Fiscal ─────────────────────
        print("⏳ Passo 17: Checkbox Anexo NF...")
        actions.send_keys(Keys.TAB).perform(); time.sleep(0.3)
        actions.send_keys(Keys.SPACE).perform(); time.sleep(0.5)

        # ── 18. ANEXO PDF ────────────────────────────────────────
        if pdf_existe:
            print("⏳ Passo 18: Anexando PDF...")
            attachment_input = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.ID, "request-attachments"))
            )
            driver.execute_script(
                "arguments[0].style.display='block';"
                "arguments[0].style.visibility='visible';"
                "arguments[0].removeAttribute('hidden');",
                attachment_input
            )
            attachment_input.send_keys(pdf_path)
            time.sleep(5)
        else:
            print("⚠️ PDF não encontrado, seguindo sem anexo.")

        # ── ENVIO ────────────────────────────────────────────────
        print("🚀 Enviando formulário N1...")
        envio_sucesso = False
        for metodo in range(1, 4):
            try:
                if metodo == 1:
                    btn = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//input[@type='submit']"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    time.sleep(0.5)
                    btn.click()
                elif metodo == 2:
                    btn = driver.find_element(By.XPATH, "//input[@type='submit']")
                    driver.execute_script("arguments[0].click();", btn)
                elif metodo == 3:
                    actions.send_keys(Keys.TAB).send_keys(Keys.TAB).send_keys(Keys.ENTER).perform()
                envio_sucesso = True
                print(f"   ✅ Enviado (método {metodo})")
                break
            except Exception:
                continue

        if not envio_sucesso:
            raise Exception("❌ Não conseguiu enviar o formulário!")

        print("⏳ Aguardando confirmação...")
        WebDriverWait(driver, 15).until(lambda d: "/requests/" in d.current_url)
        link_final = driver.current_url
        id_match = re.search(r"/requests/(\d+)", link_final)
        id_extraido = id_match.group(1) if id_match else ""

        print(f"✅ SUCESSO! ID: {id_extraido}")
        print(f"🔗 LINK: {link_final}")

        # ── Salvar resultado na planilha ─────────────────────────
        ws.range(f"V{row_index + 2}").value = id_extraido
        cell = ws.range(f"W{row_index + 2}")
        cell.value = "Abrir chamado"
        cell.api.Hyperlinks.Add(Anchor=cell.api, Address=link_final, TextToDisplay="Abrir chamado")
        ws.range(f"AE{row_index + 2}").value = "Feito pela automação N1"
        ws.range(f"AM{row_index + 2}").value = datetime.now().date()
        wb.save()

        driver.get(FORM_URL)
        time.sleep(2)
        return True, driver

    except Exception as e:
        print(f"❌ ERRO NA LINHA {row_index + 2}: {str(e)[:200]}")
        return False, None

# ================================================================
# FLUXO PRINCIPAL
# ================================================================
def main():
    print("🚀 Iniciando automação N1 com Chrome...")
    driver = inicializar_driver()
    driver = realizar_login(driver, LOGIN_FIXO)

    print("📊 Abrindo planilha Excel...")
    app = xw.App(visible=False)
    app.display_alerts   = False
    app.screen_updating  = False
    wb = None
    try:
        wb = app.books.open(EXCEL_PATH)
        ws = wb.sheets['SOLICITAÇÃO DE PAGAMENTO']
        ultima_linha = ws.range('A' + str(ws.cells.last_cell.row)).end('up').row
        df = ws.range(f"A1:AF{ultima_linha}").options(pd.DataFrame, header=1, index=False).value

        # ── Filtra linhas N1 ─────────────────────────────────────
        # Condição: coluna "Abrir ticket?" == "N2"  ← conforme instrução
        linhas_filtradas = df[df["Abrir ticket?"].astype(str).str.upper() == "N2"]
        total = len(linhas_filtradas)
        print(f"📊 Total de linhas N1 a processar: {total}\n")

        sucesso_count = 0
        for i, (idx, row) in enumerate(linhas_filtradas.iterrows(), 1):
            print(f"{'#'*60}\n📍 PROCESSANDO N1 {i}/{total}\n{'#'*60}")
            sucesso, driver_resultado = abrir_chamado_n1(idx, row, driver, wb, ws)
            if sucesso:
                driver = driver_resultado
                sucesso_count += 1
            else:
                print("🔄 Reiniciando driver após falha...")
                try: driver.quit()
                except: pass
                time.sleep(3)
                driver = inicializar_driver()
                driver = realizar_login(driver, LOGIN_FIXO)

        print(f"\n{'='*60}\n📊 RESUMO FINAL N1\n{'='*60}")
        print(f"✅ Sucessos: {sucesso_count} | ❌ Falhas: {total - sucesso_count}")

    finally:
        try:
            if wb: wb.close()
            app.quit()
            driver.quit()
        except: pass

    print("\n✅ AUTOMAÇÃO N1 FINALIZADA!")

# ---- Execução imediata (Jupyter) ----
main()

Instalando python-dateutil...
✓ Dependências checadas.
🚀 Iniciando automação N1 com Chrome...


There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


✅ Driver Chrome inicializado com sucesso

                 VERIFICANDO ESTADO DE LOGIN
✅ Sessão já ativa. Login ignorado.
📊 Abrindo planilha Excel...
📊 Total de linhas N1 a processar: 1

############################################################
📍 PROCESSANDO N1 1/1
############################################################

🟩 N1 — PROCESSANDO LINHA 2728
📄 NF: 126 | 🏪 Loja: 7181 | 🏗️ Projeto: VIRADA DE BANDEIRA
🏢 Fornecedor: JRRG | 💰 Valor: 21.500,00
🏦 Filial: 7181 | 🚩 Bandeira: EF | 📋 Ordem: 606076
📅 Vencimento: 05/05/2026
📁 PDF: ❌ Não encontrado

⏳ Passo 1: Serviços financeiros → Regularização de Notas Fiscais (4↓)...
⏳ Passo 2: Regularização NF → Lançamento Reg. NF de Serviço (1↓)...
⏳ Passo 3: Setor de Aprovação → Obras...
⚠️ Campo 'Setor de Aprovação' não encontrado. Seguindo sem preencher.
⏳ Passo 4: Centro de Custo → 7181...
⚠️ Campo 'Centro de Custo' não encontrado. Seguindo sem preencher.
⏳ Passo 5: Forma de Pagamento → Crédito em Conta (2↓)...
❌ ERRO NA LINHA 2728: Messag

In [ ]:
# ===========================================
# Automação Zendesk — N1 v2
# Preenchimento direto por ID (sem setas!)
# IDs extraídos do HTML real do formulário
# ===========================================

import sys, subprocess, importlib.util
def ensure_installed(*pkgs):
    for p in pkgs:
        if importlib.util.find_spec(p) is None:
            print(f"Instalando {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])
    print("✓ Dependências checadas.")

ensure_installed("pandas", "xlwings", "selenium", "python-dateutil", "openpyxl")

import os, time, re, getpass
import pandas as pd
import xlwings as xw
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from datetime import datetime, timedelta
from dateutil import parser

# ================================================================
# CONFIGURAÇÕES GERAIS
# ================================================================
LOGIN_FIXO   = "126815@pmenos.com.br"
EXCEL_PATH   = r"C:\Users\126815\OneDrive - paguemenos.com.br\ENGENHARIA - OBRAS - Documentos\BANCO DE DADOS - POWER BI\BI - OUTROS\BASE CONTROLE DE PAGAMENTOS.xlsx"
FORM_URL     = "https://portaldeservicos.pmenos.com.br/hc/pt-br/requests/new?ticket_form_id=41684359104269"
BASE_PDF_DIR = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

# Campos FIXOS do N1
SETOR_APROVACAO   = "Obras"
LOCAL_ATENDIMENTO_VALUE = "local_atendimento_matriz"   # value do tagger
MATERIAL_VALUE    = "material_ativo"                   # value do tagger
COD_MATERIAL      = "20103"
BANDEIRA_LIMITE   = 7000  # < 7000 → PM | >= 7000 → EF

# IDs dos campos no formulário (extraídos do HTML real)
ID_SERVICOS_FINANCEIROS = "request_custom_fields_41451492502157"
ID_REGULARIZACAO_NF     = "request_custom_fields_29043069695885"
ID_SETOR_APROVACAO      = "request_custom_fields_40482316246413"
ID_CENTRO_CUSTO         = "request_custom_fields_33516072359821"
ID_FORMA_PAGAMENTO      = "request_custom_fields_26503910558477"
ID_ORDEM                = "request_custom_fields_26503910303629"
ID_VALOR                = "request_custom_fields_27564560478221"
ID_DATA_VENCIMENTO      = "request_custom_fields_26503881758989"
ID_COD_FORNECEDOR       = "request_custom_fields_40482332756237"
ID_BANDEIRA             = "request_custom_fields_40482252411533"
ID_FORNECEDOR           = "request_custom_fields_31391155795853"
ID_MATERIAL             = "request_custom_fields_28307534550285"
ID_LOCAL_ATENDIMENTO    = "request_custom_fields_26503897088141"
ID_ASSUNTO              = "request_subject"
ID_DESCRICAO            = "request_description"
ID_CHECKBOX_NF          = "request_custom_fields_26503916779021"

# Perfil Chrome
CHROME_PROFILE_PATH = os.path.join(os.getcwd(), "ChromeProfile")
os.makedirs(CHROME_PROFILE_PATH, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"❌ Planilha não encontrada:\n   {EXCEL_PATH}")

# ================================================================
# FUNÇÕES DE PREENCHIMENTO POR ID
# ================================================================

def set_tagger_value(driver, field_id: str, value: str):
    """
    Campos 'tagger' do Zendesk usam um <input type='hidden'> com data-tagger.
    O select2 atualiza automaticamente quando disparamos o evento 'change'.
    """
    js = f"""
    var el = document.getElementById('{field_id}');
    if (el) {{
        el.value = '{value}';
        el.dispatchEvent(new Event('change', {{ bubbles: true }}));
        // Também dispara no select2 associado
        var sel = document.querySelector('select.custom-select-field-{field_id}');
        if (sel) {{
            sel.value = '{value}';
            sel.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
    }}
    """
    driver.execute_script(js)
    time.sleep(0.8)  # aguarda a UI processar campos condicionais


def set_input_value(driver, field_id: str, value: str):
    """Preenche input ou textarea pelo ID, limpando antes."""
    try:
        elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, field_id))
        )
        driver.execute_script("arguments[0].value = '';", elem)
        elem.click()
        elem.send_keys(value)
    except Exception as e:
        print(f"⚠️ Erro ao preencher #{field_id}: {e}")


def set_datepicker_value(driver, field_id: str, date_str: str):
    """
    Campos datepicker têm dois inputs: um visível (class=datepicker) e um oculto (data-format=YYYY-MM-DD).
    Precisamos preencher o oculto no formato YYYY-MM-DD e o visível com a data exibida.
    date_str: formato DD/MM/YYYY
    """
    try:
        # Converter para YYYY-MM-DD
        d = parser.parse(date_str, dayfirst=True)
        iso = d.strftime("%Y-%m-%d")
        display = d.strftime("%d/%m/%Y")

        # Preenche o campo oculto (que tem o ID real)
        js = f"""
        var hidden = document.getElementById('{field_id}');
        if (hidden) {{
            hidden.value = '{iso}';
            hidden.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
        """
        driver.execute_script(js)

        # Preenche o campo visível (datepicker) — irmão anterior ao campo oculto
        js2 = f"""
        var hidden = document.getElementById('{field_id}');
        if (hidden) {{
            var prev = hidden.previousElementSibling;
            if (prev && prev.classList.contains('datepicker')) {{
                prev.value = '{display}';
                prev.dispatchEvent(new Event('change', {{ bubbles: true }}));
            }}
        }}
        """
        driver.execute_script(js2)
        time.sleep(0.5)
    except Exception as e:
        print(f"⚠️ Erro ao preencher datepicker #{field_id}: {e}")


def check_checkbox(driver, field_id: str):
    """Marca um checkbox pelo ID."""
    try:
        js = f"""
        var cb = document.getElementById('{field_id}');
        if (cb && !cb.checked) {{
            cb.checked = true;
            cb.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
        """
        driver.execute_script(js)
        time.sleep(0.3)
    except Exception as e:
        print(f"⚠️ Erro ao marcar checkbox #{field_id}: {e}")


def wait_for_field_visible(driver, field_id: str, timeout=8) -> bool:
    """Aguarda um campo ficar visível (campos condicionais do Zendesk)."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.find_element(By.ID, field_id).is_displayed()
        )
        return True
    except Exception:
        return False

# ================================================================
# FUNÇÕES DE LOGIN
# ================================================================
def inicializar_driver():
    chrome_options = Options()
    # Descomente na 1ª execução para autenticar com MFA:
    # chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument(f"--user-data-dir={CHROME_PROFILE_PATH}")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    )
    chrome_options.add_experimental_option('excludeSwitches', ['enable-logging', 'enable-automation'])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": "C:/temp",
        "download.prompt_for_download": False,
        "profile.default_content_setting_values.notifications": 2,
    })

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(45)
    driver.implicitly_wait(10)
    driver.get(FORM_URL)
    print("✅ Chrome inicializado.")
    return driver


def realizar_login(driver, login_fixo):
    print("\n=== VERIFICANDO LOGIN ===")
    try:
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "request_subject")))
        print("✅ Sessão já ativa.")
        return driver
    except:
        print("ℹ️ Fazendo login...")

    senha = getpass.getpass(f"🔑 Senha para {login_fixo}: ")

    sels_login = [(By.ID, "identifierId"), (By.NAME, "loginfmt"), (By.ID, "i0116"), (By.CSS_SELECTOR, "input[type='email']")]
    for by, sel in sels_login:
        try:
            el = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((by, sel)))
            el.clear(); el.send_keys(login_fixo); el.send_keys(Keys.ENTER)
            print("✅ Login inserido."); time.sleep(2); break
        except: continue

    sels_senha = [(By.NAME, "password"), (By.NAME, "passwd"), (By.ID, "i0118"), (By.CSS_SELECTOR, "input[type='password']")]
    for by, sel in sels_senha:
        try:
            el = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, sel)))
            el.send_keys(senha); time.sleep(1); el.send_keys(Keys.ENTER)
            print("✅ Senha inserida."); time.sleep(3); break
        except: continue

    # MFA
    sels_token = [(By.ID, "idTxtBx_SAOTCC_OTC"), (By.NAME, "otc"),
                  (By.CSS_SELECTOR, "input[aria-label='Código de verificação']")]
    campo_token = None
    for _ in range(20):
        for by, sel in sels_token:
            try:
                el = driver.find_element(by, sel)
                if el.is_displayed(): campo_token = el; break
            except: pass
        if campo_token: break
        time.sleep(1)

    if campo_token:
        token = input("🔒 Token MFA: ")
        campo_token.send_keys(token); time.sleep(1); campo_token.send_keys(Keys.ENTER)
        print("✅ Token inserido.")

    # Continuar conectado
    for by, sel in [(By.ID, "idSIButton9"), (By.CSS_SELECTOR, "input[value='Sim']")]:
        try:
            btn = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, sel)))
            btn.click(); print("✅ 'Continuar conectado' confirmado."); break
        except: pass

    print("=== LOGIN CONCLUÍDO ===")
    time.sleep(5)
    return driver

# ================================================================
# ABERTURA DO CHAMADO N1
# ================================================================
def abrir_chamado_n1(row_index, row_data, driver, wb, ws):
    # ── Leitura dos dados ────────────────────────────────────────
    assunto = str(row_data["ASSUNTO"]).strip()

    # Ordem
    ordem_raw = row_data.get("ORDEM", "")
    if pd.notna(ordem_raw) and str(ordem_raw).replace('.','',1).isdigit():
        ordem = str(int(float(ordem_raw)))
    else:
        ordem = str(ordem_raw).strip() if pd.notna(ordem_raw) else ""

    # Filial → Centro de Custo
    filial_raw = row_data.get("FILIAL", "")
    if pd.notna(filial_raw) and str(filial_raw).replace('.','',1).isdigit():
        filial = str(int(float(filial_raw)))
    else:
        filial = str(filial_raw).strip() if pd.notna(filial_raw) else ""

    # Bandeira
    try:
        bandeira = "PM" if int(filial) < BANDEIRA_LIMITE else "EF"
    except:
        bandeira = "PM"

    # Valor
    valor_raw = str(row_data.get("VALOR BI", "0")).strip()
    try:
        valor_limpo = re.sub(r"[^\d.,]", "", valor_raw)
        if re.match(r"^\d{1,3}(\.\d{3})*,\d{2}$", valor_limpo):
            valor_formatado = valor_limpo
        else:
            valor_num = float(valor_limpo.replace(',', '.'))
            valor_formatado = f"{valor_num:,.2f}".replace(",","X").replace(".",",").replace("X",".")
    except:
        valor_formatado = "0,00"

    # Data de Vencimento
    venc_match = re.search(r"VENC\s*(\d{2}/\d{2}/\d{4})", assunto)
    if venc_match:
        try:
            data_vencimento = parser.parse(venc_match.group(1), dayfirst=True).strftime("%d/%m/%Y")
        except:
            data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")
    else:
        data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")

    # Extração de campos do assunto
    cd_match = re.search(r"CD\s*([A-Z0-9]{2,4})", assunto, re.IGNORECASE)
    if cd_match:
        loja    = f"CD {cd_match.group(1)}"
        pm      = re.match(r"(.+?)\s*CD\s*[A-Z0-9]{2,4}", assunto, re.IGNORECASE)
        projeto = pm.group(1).strip() if pm else ""
    else:
        pl = re.match(r"([A-Z\s]+)\s+(\d{3,4})", assunto)
        projeto = pl.group(1).strip() if pl else ""
        loja    = pl.group(2).strip() if pl else ""

    if not projeto:
        fb = re.match(r"(.+?)(?=\s*-\s*|\s*N[°º])", assunto, re.IGNORECASE)
        projeto = fb.group(1).strip() if fb else ""

    forn_match = re.search(r"-\s*([A-ZÀ-Úa-z0-9 ]+?)(?=\s*-|\s*N[°º])", assunto)
    fornecedor = forn_match.group(1).strip() if forn_match else ""

    nf_match = re.search(r"N[°º]?\s*(\d+)", assunto)
    nf = nf_match.group(1) if nf_match else ""

    # Fornecedor da base tem prioridade
    fornecedor_base = ""
    try:
        if "FORNECEDOR" in row_data.index and pd.notna(row_data["FORNECEDOR"]):
            fornecedor_base = str(row_data["FORNECEDOR"]).strip()
    except: pass
    fornecedor_final = fornecedor_base or fornecedor

    # Cod. Fornecedor (opcional)
    cod_fornecedor = ""
    try:
        if "COD_FORNECEDOR" in row_data.index and pd.notna(row_data["COD_FORNECEDOR"]):
            cod_fornecedor = str(row_data["COD_FORNECEDOR"]).strip()
    except: pass

    # Descrição
    descricao_final = f"{assunto} | ORDEM: {ordem}"

    # PDF
    if loja.startswith("CD "):
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - {loja}.pdf")
    else:
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - LJ {loja}.pdf")
    pdf_existe = os.path.exists(pdf_path)

    print(f"\n{'='*60}")
    print(f"🟩 N1 — LINHA {row_index + 2}")
    print(f"   NF: {nf} | Loja: {loja} | Projeto: {projeto}")
    print(f"   Fornecedor: {fornecedor_final} | Valor: {valor_formatado}")
    print(f"   Filial: {filial} | Bandeira: {bandeira} | Ordem: {ordem}")
    print(f"   Vencimento: {data_vencimento}")
    print(f"   PDF: {'✅' if pdf_existe else '❌ não encontrado'}")
    print(f"{'='*60}")

    try:
        # Aguarda o formulário carregar
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, ID_ASSUNTO))
        )
        time.sleep(1)

        # ── 1. Serviços Financeiros ao Fornecedor ───────────────
        print("⏳ 1. Serviços financeiros → Regularização de Notas Fiscais")
        set_tagger_value(driver, ID_SERVICOS_FINANCEIROS,
                         "otc_servicos_financeiros_fornecedor_regularizacao_de_notas_fiscais")
        time.sleep(1.5)  # espera campos condicionais aparecerem

        # ── 2. Regularização de Notas Fiscais ───────────────────
        print("⏳ 2. Regularização NF → Lançamento Reg. NF de Serviço")
        wait_for_field_visible(driver, ID_REGULARIZACAO_NF)
        set_tagger_value(driver, ID_REGULARIZACAO_NF,
                         "regularizacao_lancamento_regularizacao_nota_fiscal_servico")
        time.sleep(1.5)  # espera campos condicionais aparecerem

        # ── 3. Setor de Aprovação ────────────────────────────────
        print(f"⏳ 3. Setor de Aprovação → {SETOR_APROVACAO}")
        wait_for_field_visible(driver, ID_SETOR_APROVACAO)
        set_input_value(driver, ID_SETOR_APROVACAO, SETOR_APROVACAO)

        # ── 4. Centro de Custo ───────────────────────────────────
        print(f"⏳ 4. Centro de Custo → {filial}")
        wait_for_field_visible(driver, ID_CENTRO_CUSTO)
        set_input_value(driver, ID_CENTRO_CUSTO, filial)

        # ── 5. Forma de Pagamento → Crédito em Conta ────────────
        print("⏳ 5. Forma de Pagamento → Crédito em Conta")
        wait_for_field_visible(driver, ID_FORMA_PAGAMENTO)
        set_tagger_value(driver, ID_FORMA_PAGAMENTO, "forma_pagamento_credito_em_conta")

        # ── 6. Ordem ─────────────────────────────────────────────
        print(f"⏳ 6. Ordem → {ordem}")
        wait_for_field_visible(driver, ID_ORDEM)
        set_input_value(driver, ID_ORDEM, ordem)

        # ── 7. Valor ─────────────────────────────────────────────
        print(f"⏳ 7. Valor → {valor_formatado}")
        wait_for_field_visible(driver, ID_VALOR)
        set_input_value(driver, ID_VALOR, valor_formatado)

        # ── 8. Data de Vencimento ────────────────────────────────
        print(f"⏳ 8. Data de Vencimento → {data_vencimento}")
        wait_for_field_visible(driver, ID_DATA_VENCIMENTO)
        set_datepicker_value(driver, ID_DATA_VENCIMENTO, data_vencimento)

        # ── 9. Cod. Fornecedor (opcional) ────────────────────────
        if cod_fornecedor:
            print(f"⏳ 9. Cod.Fornecedor → {cod_fornecedor}")
            wait_for_field_visible(driver, ID_COD_FORNECEDOR)
            set_input_value(driver, ID_COD_FORNECEDOR, cod_fornecedor)

        # ── 10. Bandeira ─────────────────────────────────────────
        print(f"⏳ 10. Bandeira → {bandeira}")
        wait_for_field_visible(driver, ID_BANDEIRA)
        set_input_value(driver, ID_BANDEIRA, bandeira)

        # ── 11. Fornecedor ───────────────────────────────────────
        if fornecedor_final:
            print(f"⏳ 11. Fornecedor → {fornecedor_final}")
            wait_for_field_visible(driver, ID_FORNECEDOR)
            set_input_value(driver, ID_FORNECEDOR, fornecedor_final)

        # ── 12. Material → Material Ativo ────────────────────────
        print("⏳ 12. Material → Material Ativo")
        wait_for_field_visible(driver, ID_MATERIAL)
        set_tagger_value(driver, ID_MATERIAL, MATERIAL_VALUE)

        # ── 13. Cód Material → sempre na descrição ──────────────
        print(f"⏳ 13. Cód Material {COD_MATERIAL} → incluído na descrição")
        descricao_final += f" | CÓD MATERIAL: {COD_MATERIAL}"

        # ── 14. Local de Atendimento → Matriz ───────────────────
        print("⏳ 14. Local de Atendimento → Matriz")
        wait_for_field_visible(driver, ID_LOCAL_ATENDIMENTO)
        set_tagger_value(driver, ID_LOCAL_ATENDIMENTO, LOCAL_ATENDIMENTO_VALUE)

        # ── 15. Assunto ──────────────────────────────────────────
        print("⏳ 15. Assunto")
        set_input_value(driver, ID_ASSUNTO, assunto)

        # ── 16. Descrição ────────────────────────────────────────
        print("⏳ 16. Descrição")
        set_input_value(driver, ID_DESCRICAO, descricao_final)

        # ── 17. Checkbox [Anexo] Nota Fiscal ─────────────────────
        print("⏳ 17. Checkbox Nota Fiscal")
        check_checkbox(driver, ID_CHECKBOX_NF)

        # ── 18. Anexo PDF ────────────────────────────────────────
        if pdf_existe:
            print("⏳ 18. Anexando PDF...")
            attachment_input = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.ID, "request-attachments"))
            )
            driver.execute_script(
                "arguments[0].style.display='block';"
                "arguments[0].style.visibility='visible';"
                "arguments[0].removeAttribute('hidden');",
                attachment_input
            )
            attachment_input.send_keys(pdf_path)
            time.sleep(5)
        else:
            print("⚠️ PDF não encontrado — seguindo sem anexo.")

        # ── ENVIO ────────────────────────────────────────────────
        print("🚀 Enviando formulário N1...")
        envio_sucesso = False
        for metodo in range(1, 4):
            try:
                if metodo == 1:
                    btn = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//input[@type='submit']"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    time.sleep(0.5); btn.click()
                elif metodo == 2:
                    btn = driver.find_element(By.XPATH, "//input[@type='submit']")
                    driver.execute_script("arguments[0].click();", btn)
                elif metodo == 3:
                    driver.execute_script(
                        "document.querySelector('input[type=submit]').click();"
                    )
                envio_sucesso = True
                print(f"   ✅ Enviado (método {metodo})")
                break
            except Exception as e:
                print(f"   ⚠️ Método {metodo} falhou: {e}")
                continue

        if not envio_sucesso:
            raise Exception("❌ Não conseguiu enviar!")

        print("⏳ Aguardando confirmação...")
        WebDriverWait(driver, 20).until(lambda d: "/requests/" in d.current_url)
        link_final = driver.current_url
        id_match = re.search(r"/requests/(\d+)", link_final)
        id_extraido = id_match.group(1) if id_match else ""

        print(f"✅ SUCESSO! ID: {id_extraido}")
        print(f"🔗 {link_final}")

        # Salva na planilha
        ws.range(f"V{row_index + 2}").value = id_extraido
        cell = ws.range(f"W{row_index + 2}")
        cell.value = "Abrir chamado"
        cell.api.Hyperlinks.Add(Anchor=cell.api, Address=link_final, TextToDisplay="Abrir chamado")
        ws.range(f"AE{row_index + 2}").value = "Feito pela automação N1"
        ws.range(f"AM{row_index + 2}").value = datetime.now().date()
        wb.save()

        driver.get(FORM_URL)
        time.sleep(2)
        return True, driver

    except Exception as e:
        print(f"❌ ERRO NA LINHA {row_index + 2}: {str(e)[:300]}")
        return False, None

# ================================================================
# FLUXO PRINCIPAL
# ================================================================
def main():
    print("🚀 Iniciando automação N1...")
    driver = inicializar_driver()
    driver = realizar_login(driver, LOGIN_FIXO)

    print("📊 Abrindo planilha...")
    app = xw.App(visible=False)
    app.display_alerts  = False
    app.screen_updating = False
    wb = None
    try:
        wb = app.books.open(EXCEL_PATH)
        ws = wb.sheets['SOLICITAÇÃO DE PAGAMENTO']
        ultima_linha = ws.range('A' + str(ws.cells.last_cell.row)).end('up').row
        df = ws.range(f"A1:AF{ultima_linha}").options(pd.DataFrame, header=1, index=False).value

        # Filtra linhas N1 (coluna "Abrir ticket?" == "N2")
        linhas_filtradas = df[df["Abrir ticket?"].astype(str).str.upper() == "N2"]
        total = len(linhas_filtradas)
        print(f"📊 {total} linha(s) a processar.\n")

        sucesso_count = 0
        for i, (idx, row) in enumerate(linhas_filtradas.iterrows(), 1):
            print(f"\n{'#'*60}\n📍 {i}/{total}\n{'#'*60}")
            sucesso, driver_res = abrir_chamado_n1(idx, row, driver, wb, ws)
            if sucesso:
                driver = driver_res
                sucesso_count += 1
            else:
                print("🔄 Reiniciando driver...")
                try: driver.quit()
                except: pass
                time.sleep(3)
                driver = inicializar_driver()
                driver = realizar_login(driver, LOGIN_FIXO)

        print(f"\n{'='*60}")
        print(f"📊 RESUMO N1: ✅ {sucesso_count} | ❌ {total - sucesso_count}")

    finally:
        try:
            if wb: wb.close()
            app.quit()
            driver.quit()
        except: pass

    print("\n✅ AUTOMAÇÃO N1 FINALIZADA!")

main()

Instalando python-dateutil...
✓ Dependências checadas.


In [1]:
# ===========================================
# Automação Zendesk — N1 v2
# Preenchimento direto por ID (sem setas!)
# IDs extraídos do HTML real do formulário
# ===========================================

import sys, subprocess, importlib.util
def ensure_installed(*pkgs):
    for p in pkgs:
        if importlib.util.find_spec(p) is None:
            print(f"Instalando {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])
    print("✓ Dependências checadas.")

ensure_installed("pandas", "xlwings", "selenium", "python-dateutil", "openpyxl")

import os, time, re, getpass
import pandas as pd
import xlwings as xw
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from datetime import datetime, timedelta
from dateutil import parser

# ================================================================
# CONFIGURAÇÕES GERAIS
# ================================================================
LOGIN_FIXO   = "126815@pmenos.com.br"
EXCEL_PATH   = r"C:\Users\126815\OneDrive - paguemenos.com.br\ENGENHARIA - OBRAS - Documentos\BANCO DE DADOS - POWER BI\BI - OUTROS\BASE CONTROLE DE PAGAMENTOS.xlsx"
FORM_URL     = "https://portaldeservicos.pmenos.com.br/hc/pt-br/requests/new?ticket_form_id=41684359104269"
BASE_PDF_DIR = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

# Campos FIXOS do N1
SETOR_APROVACAO   = "Obras"
LOCAL_ATENDIMENTO_VALUE = "local_atendimento_matriz"   # value do tagger
MATERIAL_VALUE    = "material_ativo"                   # value do tagger
COD_MATERIAL      = "20272"
BANDEIRA_LIMITE   = 7000  # < 7000 → PM | >= 7000 → EF

# IDs dos campos no formulário (extraídos do HTML real)
ID_SERVICOS_FINANCEIROS = "request_custom_fields_41451492502157"
ID_REGULARIZACAO_NF     = "request_custom_fields_29043069695885"
ID_SETOR_APROVACAO      = "request_custom_fields_40482316246413"
ID_CENTRO_CUSTO         = "request_custom_fields_33516072359821"
ID_FORMA_PAGAMENTO      = "request_custom_fields_26503910558477"
ID_ORDEM                = "request_custom_fields_26503910303629"
ID_VALOR                = "request_custom_fields_27564560478221"
ID_DATA_VENCIMENTO      = "request_custom_fields_26503881758989"
ID_COD_FORNECEDOR       = "request_custom_fields_40482332756237"
ID_BANDEIRA             = "request_custom_fields_40482252411533"
ID_FORNECEDOR           = "request_custom_fields_31391155795853"
ID_MATERIAL             = "request_custom_fields_28307534550285"
ID_LOCAL_ATENDIMENTO    = "request_custom_fields_26503897088141"
ID_ASSUNTO              = "request_subject"
ID_DESCRICAO            = "request_description"
ID_CHECKBOX_NF          = "request_custom_fields_26503916779021"

# Perfil Chrome
CHROME_PROFILE_PATH = os.path.join(os.getcwd(), "ChromeProfile")
os.makedirs(CHROME_PROFILE_PATH, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"❌ Planilha não encontrada:\n   {EXCEL_PATH}")

# ================================================================
# FUNÇÕES DE PREENCHIMENTO POR ID
# ================================================================

def set_tagger_value(driver, field_id: str, value: str):
    """
    Campos 'tagger' do Zendesk usam um <input type='hidden'> com data-tagger.
    O select2 atualiza automaticamente quando disparamos o evento 'change'.
    """
    js = f"""
    var el = document.getElementById('{field_id}');
    if (el) {{
        el.value = '{value}';
        el.dispatchEvent(new Event('change', {{ bubbles: true }}));
        // Também dispara no select2 associado
        var sel = document.querySelector('select.custom-select-field-{field_id}');
        if (sel) {{
            sel.value = '{value}';
            sel.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
    }}
    """
    driver.execute_script(js)
    time.sleep(0.8)  # aguarda a UI processar campos condicionais


def set_input_value(driver, field_id: str, value: str):
    """Preenche input ou textarea pelo ID, limpando antes."""
    try:
        elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, field_id))
        )
        driver.execute_script("arguments[0].value = '';", elem)
        elem.click()
        elem.send_keys(value)
    except Exception as e:
        print(f"⚠️ Erro ao preencher #{field_id}: {e}")


def set_datepicker_value(driver, field_id: str, date_str: str):
    """
    Campos datepicker têm dois inputs: um visível (class=datepicker) e um oculto (data-format=YYYY-MM-DD).
    Precisamos preencher o oculto no formato YYYY-MM-DD e o visível com a data exibida.
    date_str: formato DD/MM/YYYY
    """
    try:
        # Converter para YYYY-MM-DD
        d = parser.parse(date_str, dayfirst=True)
        iso = d.strftime("%Y-%m-%d")
        display = d.strftime("%d/%m/%Y")

        # Preenche o campo oculto (que tem o ID real)
        js = f"""
        var hidden = document.getElementById('{field_id}');
        if (hidden) {{
            hidden.value = '{iso}';
            hidden.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
        """
        driver.execute_script(js)

        # Preenche o campo visível (datepicker) — irmão anterior ao campo oculto
        js2 = f"""
        var hidden = document.getElementById('{field_id}');
        if (hidden) {{
            var prev = hidden.previousElementSibling;
            if (prev && prev.classList.contains('datepicker')) {{
                prev.value = '{display}';
                prev.dispatchEvent(new Event('change', {{ bubbles: true }}));
            }}
        }}
        """
        driver.execute_script(js2)
        time.sleep(0.5)
    except Exception as e:
        print(f"⚠️ Erro ao preencher datepicker #{field_id}: {e}")


def check_checkbox(driver, field_id: str):
    """Marca um checkbox pelo ID."""
    try:
        js = f"""
        var cb = document.getElementById('{field_id}');
        if (cb && !cb.checked) {{
            cb.checked = true;
            cb.dispatchEvent(new Event('change', {{ bubbles: true }}));
        }}
        """
        driver.execute_script(js)
        time.sleep(0.3)
    except Exception as e:
        print(f"⚠️ Erro ao marcar checkbox #{field_id}: {e}")


def wait_for_field_visible(driver, field_id: str, timeout=8) -> bool:
    """Aguarda um campo ficar visível (campos condicionais do Zendesk)."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.find_element(By.ID, field_id).is_displayed()
        )
        return True
    except Exception:
        return False

# ================================================================
# FUNÇÕES DE LOGIN
# ================================================================
def inicializar_driver():
    chrome_options = Options()
    # Descomente na 1ª execução para autenticar com MFA:
    # chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    # ✅ FIX: Desativa AI Mode / AI Overviews do Chrome (resolve bug do Chrome 147+)
    chrome_options.add_argument("--disable-features=AIMode,SearchAIMode,AIOverviews,SearchGenerativeExperience")

    chrome_options.add_argument(f"--user-data-dir={CHROME_PROFILE_PATH}")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    )
    chrome_options.add_experimental_option('excludeSwitches', ['enable-logging', 'enable-automation'])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": "C:/temp",
        "download.prompt_for_download": False,
        "profile.default_content_setting_values.notifications": 2,
    })

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(45)
    driver.implicitly_wait(10)
    driver.get(FORM_URL)
    print("✅ Chrome inicializado.")
    return driver


def realizar_login(driver, login_fixo):
    print("\n=== VERIFICANDO LOGIN ===")
    try:
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "request_subject")))
        print("✅ Sessão já ativa.")
        return driver
    except:
        print("ℹ️ Fazendo login...")

    senha = getpass.getpass(f"🔑 Senha para {login_fixo}: ")

    sels_login = [(By.ID, "identifierId"), (By.NAME, "loginfmt"), (By.ID, "i0116"), (By.CSS_SELECTOR, "input[type='email']")]
    for by, sel in sels_login:
        try:
            el = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((by, sel)))
            el.clear(); el.send_keys(login_fixo); el.send_keys(Keys.ENTER)
            print("✅ Login inserido."); time.sleep(2); break
        except: continue

    sels_senha = [(By.NAME, "password"), (By.NAME, "passwd"), (By.ID, "i0118"), (By.CSS_SELECTOR, "input[type='password']")]
    for by, sel in sels_senha:
        try:
            el = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, sel)))
            el.send_keys(senha); time.sleep(1); el.send_keys(Keys.ENTER)
            print("✅ Senha inserida."); time.sleep(3); break
        except: continue

    # MFA
    sels_token = [(By.ID, "idTxtBx_SAOTCC_OTC"), (By.NAME, "otc"),
                  (By.CSS_SELECTOR, "input[aria-label='Código de verificação']")]
    campo_token = None
    for _ in range(20):
        for by, sel in sels_token:
            try:
                el = driver.find_element(by, sel)
                if el.is_displayed(): campo_token = el; break
            except: pass
        if campo_token: break
        time.sleep(1)

    if campo_token:
        token = input("🔒 Token MFA: ")
        campo_token.send_keys(token); time.sleep(1); campo_token.send_keys(Keys.ENTER)
        print("✅ Token inserido.")

    # Continuar conectado
    for by, sel in [(By.ID, "idSIButton9"), (By.CSS_SELECTOR, "input[value='Sim']")]:
        try:
            btn = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, sel)))
            btn.click(); print("✅ 'Continuar conectado' confirmado."); break
        except: pass

    print("=== LOGIN CONCLUÍDO ===")
    time.sleep(5)
    return driver

# ================================================================
# ABERTURA DO CHAMADO N1
# ================================================================
def abrir_chamado_n1(row_index, row_data, driver, wb, ws):
    # ── Leitura dos dados ────────────────────────────────────────
    assunto = str(row_data["ASSUNTO"]).strip()

    # Ordem
    ordem_raw = row_data.get("ORDEM", "")
    if pd.notna(ordem_raw) and str(ordem_raw).replace('.','',1).isdigit():
        ordem = str(int(float(ordem_raw)))
    else:
        ordem = str(ordem_raw).strip() if pd.notna(ordem_raw) else ""

    # Filial → Centro de Custo
    filial_raw = row_data.get("FILIAL", "")
    if pd.notna(filial_raw) and str(filial_raw).replace('.','',1).isdigit():
        filial = str(int(float(filial_raw)))
    else:
        filial = str(filial_raw).strip() if pd.notna(filial_raw) else ""

    # Bandeira
    try:
        bandeira = "PM" if int(filial) < BANDEIRA_LIMITE else "EF"
    except:
        bandeira = "PM"

    # Valor
    valor_raw = str(row_data.get("VALOR BI", "0")).strip()
    try:
        valor_limpo = re.sub(r"[^\d.,]", "", valor_raw)
        if re.match(r"^\d{1,3}(\.\d{3})*,\d{2}$", valor_limpo):
            valor_formatado = valor_limpo
        else:
            valor_num = float(valor_limpo.replace(',', '.'))
            valor_formatado = f"{valor_num:,.2f}".replace(",","X").replace(".",",").replace("X",".")
    except:
        valor_formatado = "0,00"

    # Data de Vencimento
    venc_match = re.search(r"VENC\s*(\d{2}/\d{2}/\d{4})", assunto)
    if venc_match:
        try:
            data_vencimento = parser.parse(venc_match.group(1), dayfirst=True).strftime("%d/%m/%Y")
        except:
            data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")
    else:
        data_vencimento = (datetime.now() + timedelta(days=7)).strftime("%d/%m/%Y")

    # Extração de campos do assunto
    cd_match = re.search(r"CD\s*([A-Z0-9]{2,4})", assunto, re.IGNORECASE)
    if cd_match:
        loja    = f"CD {cd_match.group(1)}"
        pm      = re.match(r"(.+?)\s*CD\s*[A-Z0-9]{2,4}", assunto, re.IGNORECASE)
        projeto = pm.group(1).strip() if pm else ""
    else:
        pl = re.match(r"([A-Z\s]+)\s+(\d{3,4})", assunto)
        projeto = pl.group(1).strip() if pl else ""
        loja    = pl.group(2).strip() if pl else ""

    if not projeto:
        fb = re.match(r"(.+?)(?=\s*-\s*|\s*N[°º])", assunto, re.IGNORECASE)
        projeto = fb.group(1).strip() if fb else ""

    forn_match = re.search(r"-\s*([A-ZÀ-Úa-z0-9 ]+?)(?=\s*-|\s*N[°º])", assunto)
    fornecedor = forn_match.group(1).strip() if forn_match else ""

    nf_match = re.search(r"N[°º]?\s*(\d+)", assunto)
    nf = nf_match.group(1) if nf_match else ""

    # Fornecedor da base tem prioridade
    fornecedor_base = ""
    try:
        if "FORNECEDOR" in row_data.index and pd.notna(row_data["FORNECEDOR"]):
            fornecedor_base = str(row_data["FORNECEDOR"]).strip()
    except: pass
    fornecedor_final = fornecedor_base or fornecedor

    # Cod. Fornecedor (opcional)
    cod_fornecedor = ""
    try:
        if "COD_FORNECEDOR" in row_data.index and pd.notna(row_data["COD_FORNECEDOR"]):
            cod_fornecedor = str(row_data["COD_FORNECEDOR"]).strip()
    except: pass

    # Descrição
    descricao_final = f"{assunto} | ORDEM: {ordem}"

    # PDF
    if loja.startswith("CD "):
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - {loja}.pdf")
    else:
        pdf_path = os.path.join(BASE_PDF_DIR, projeto, fornecedor_final, f"NF {nf} - LJ {loja}.pdf")
    pdf_existe = os.path.exists(pdf_path)

    print(f"\n{'='*60}")
    print(f"🟩 N1 — LINHA {row_index + 2}")
    print(f"   NF: {nf} | Loja: {loja} | Projeto: {projeto}")
    print(f"   Fornecedor: {fornecedor_final} | Valor: {valor_formatado}")
    print(f"   Filial: {filial} | Bandeira: {bandeira} | Ordem: {ordem}")
    print(f"   Vencimento: {data_vencimento}")
    print(f"   PDF: {'✅' if pdf_existe else '❌ não encontrado'}")
    print(f"{'='*60}")

    try:
        # Aguarda o formulário carregar
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, ID_ASSUNTO))
        )
        time.sleep(1)

        # ── 1. Serviços Financeiros ao Fornecedor ───────────────
        print("⏳ 1. Serviços financeiros → Regularização de Notas Fiscais")
        set_tagger_value(driver, ID_SERVICOS_FINANCEIROS,
                         "otc_servicos_financeiros_fornecedor_regularizacao_de_notas_fiscais")
        time.sleep(1.5)  # espera campos condicionais aparecerem

        # ── 2. Regularização de Notas Fiscais ───────────────────
        print("⏳ 2. Regularização NF → Lançamento Reg. NF de Serviço")
        wait_for_field_visible(driver, ID_REGULARIZACAO_NF)
        set_tagger_value(driver, ID_REGULARIZACAO_NF,
                         "regularizacao_lancamento_regularizacao_nota_fiscal_servico")
        time.sleep(1.5)  # espera campos condicionais aparecerem

        # ── 3. Setor de Aprovação ────────────────────────────────
        print(f"⏳ 3. Setor de Aprovação → {SETOR_APROVACAO}")
        wait_for_field_visible(driver, ID_SETOR_APROVACAO)
        set_input_value(driver, ID_SETOR_APROVACAO, SETOR_APROVACAO)

        # ── 4. Centro de Custo ───────────────────────────────────
        print(f"⏳ 4. Centro de Custo → {filial}")
        wait_for_field_visible(driver, ID_CENTRO_CUSTO)
        set_input_value(driver, ID_CENTRO_CUSTO, filial)

        # ── 5. Forma de Pagamento → Crédito em Conta ────────────
        print("⏳ 5. Forma de Pagamento → Crédito em Conta")
        wait_for_field_visible(driver, ID_FORMA_PAGAMENTO)
        set_tagger_value(driver, ID_FORMA_PAGAMENTO, "forma_pagamento_credito_em_conta")

        # ── 6. Ordem ─────────────────────────────────────────────
        print(f"⏳ 6. Ordem → {ordem}")
        wait_for_field_visible(driver, ID_ORDEM)
        set_input_value(driver, ID_ORDEM, ordem)

        # ── 7. Valor ─────────────────────────────────────────────
        print(f"⏳ 7. Valor → {valor_formatado}")
        wait_for_field_visible(driver, ID_VALOR)
        set_input_value(driver, ID_VALOR, valor_formatado)

        # ── 8. Data de Vencimento ────────────────────────────────
        print(f"⏳ 8. Data de Vencimento → {data_vencimento}")
        wait_for_field_visible(driver, ID_DATA_VENCIMENTO)
        set_datepicker_value(driver, ID_DATA_VENCIMENTO, data_vencimento)

        # ── 9. Cod. Fornecedor (opcional) ────────────────────────
        if cod_fornecedor:
            print(f"⏳ 9. Cod.Fornecedor → {cod_fornecedor}")
            wait_for_field_visible(driver, ID_COD_FORNECEDOR)
            set_input_value(driver, ID_COD_FORNECEDOR, cod_fornecedor)

        # ── 10. Bandeira ─────────────────────────────────────────
        print(f"⏳ 10. Bandeira → {bandeira}")
        wait_for_field_visible(driver, ID_BANDEIRA)
        set_input_value(driver, ID_BANDEIRA, bandeira)

        # ── 11. Fornecedor ───────────────────────────────────────
        if fornecedor_final:
            print(f"⏳ 11. Fornecedor → {fornecedor_final}")
            wait_for_field_visible(driver, ID_FORNECEDOR)
            set_input_value(driver, ID_FORNECEDOR, fornecedor_final)

        # ── 12. Material → Material Ativo ────────────────────────
        print("⏳ 12. Material → Material Ativo")
        wait_for_field_visible(driver, ID_MATERIAL)
        set_tagger_value(driver, ID_MATERIAL, MATERIAL_VALUE)

        # ── 13. Cód Material → sempre na descrição ──────────────
        print(f"⏳ 13. Cód Material {COD_MATERIAL} → incluído na descrição")
        descricao_final += f" | CÓD MATERIAL: {COD_MATERIAL}"

        # ── 14. Local de Atendimento → Matriz ───────────────────
        print("⏳ 14. Local de Atendimento → Matriz")
        wait_for_field_visible(driver, ID_LOCAL_ATENDIMENTO)
        set_tagger_value(driver, ID_LOCAL_ATENDIMENTO, LOCAL_ATENDIMENTO_VALUE)

        # ── 15. Assunto ──────────────────────────────────────────
        print("⏳ 15. Assunto")
        set_input_value(driver, ID_ASSUNTO, assunto)

        # ── 16. Descrição ────────────────────────────────────────
        print("⏳ 16. Descrição")
        set_input_value(driver, ID_DESCRICAO, descricao_final)

        # ── 17. Checkbox [Anexo] Nota Fiscal ─────────────────────
        print("⏳ 17. Checkbox Nota Fiscal")
        check_checkbox(driver, ID_CHECKBOX_NF)

        # ── 18. Anexo PDF ────────────────────────────────────────
        if pdf_existe:
            print("⏳ 18. Anexando PDF...")
            attachment_input = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.ID, "request-attachments"))
            )
            driver.execute_script(
                "arguments[0].style.display='block';"
                "arguments[0].style.visibility='visible';"
                "arguments[0].removeAttribute('hidden');",
                attachment_input
            )
            attachment_input.send_keys(pdf_path)
            time.sleep(5)
        else:
            print("⚠️ PDF não encontrado — seguindo sem anexo.")

        # ── ENVIO ────────────────────────────────────────────────
        print("🚀 Enviando formulário N1...")
        envio_sucesso = False
        for metodo in range(1, 4):
            try:
                if metodo == 1:
                    btn = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//input[@type='submit']"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    time.sleep(0.5); btn.click()
                elif metodo == 2:
                    btn = driver.find_element(By.XPATH, "//input[@type='submit']")
                    driver.execute_script("arguments[0].click();", btn)
                elif metodo == 3:
                    driver.execute_script(
                        "document.querySelector('input[type=submit]').click();"
                    )
                envio_sucesso = True
                print(f"   ✅ Enviado (método {metodo})")
                break
            except Exception as e:
                print(f"   ⚠️ Método {metodo} falhou: {e}")
                continue

        if not envio_sucesso:
            raise Exception("❌ Não conseguiu enviar!")

        print("⏳ Aguardando confirmação...")
        WebDriverWait(driver, 20).until(lambda d: "/requests/" in d.current_url)
        link_final = driver.current_url
        id_match = re.search(r"/requests/(\d+)", link_final)
        id_extraido = id_match.group(1) if id_match else ""

        print(f"✅ SUCESSO! ID: {id_extraido}")
        print(f"🔗 {link_final}")

        # Salva na planilha
        ws.range(f"V{row_index + 2}").value = id_extraido
        cell = ws.range(f"W{row_index + 2}")
        cell.value = "Abrir chamado"
        cell.api.Hyperlinks.Add(Anchor=cell.api, Address=link_final, TextToDisplay="Abrir chamado")
        ws.range(f"AE{row_index + 2}").value = "Feito pela automação N1"
        ws.range(f"AM{row_index + 2}").value = datetime.now().date()
        wb.save()

        driver.get(FORM_URL)
        time.sleep(2)
        return True, driver

    except Exception as e:
        print(f"❌ ERRO NA LINHA {row_index + 2}: {str(e)[:300]}")
        return False, None

# ================================================================
# FLUXO PRINCIPAL
# ================================================================
def main():
    print("🚀 Iniciando automação N1...")
    driver = inicializar_driver()
    driver = realizar_login(driver, LOGIN_FIXO)

    print("📊 Abrindo planilha...")
    app = xw.App(visible=False)
    app.display_alerts  = False
    app.screen_updating = False
    wb = None
    try:
        wb = app.books.open(EXCEL_PATH)
        ws = wb.sheets['SOLICITAÇÃO DE PAGAMENTO']
        ultima_linha = ws.range('A' + str(ws.cells.last_cell.row)).end('up').row
        df = ws.range(f"A1:AF{ultima_linha}").options(pd.DataFrame, header=1, index=False).value

        # Filtra linhas N1 (coluna "Abrir ticket?" == "N2")
        linhas_filtradas = df[df["Abrir ticket?"].astype(str).str.upper() == "N1"]
        total = len(linhas_filtradas)
        print(f"📊 {total} linha(s) a processar.\n")

        sucesso_count = 0
        for i, (idx, row) in enumerate(linhas_filtradas.iterrows(), 1):
            print(f"\n{'#'*60}\n📍 {i}/{total}\n{'#'*60}")
            sucesso, driver_res = abrir_chamado_n1(idx, row, driver, wb, ws)
            if sucesso:
                driver = driver_res
                sucesso_count += 1
            else:
                print("🔄 Reiniciando driver...")
                try: driver.quit()
                except: pass
                time.sleep(3)
                driver = inicializar_driver()
                driver = realizar_login(driver, LOGIN_FIXO)

        print(f"\n{'='*60}")
        print(f"📊 RESUMO N1: ✅ {sucesso_count} | ❌ {total - sucesso_count}")

    finally:
        try:
            if wb: wb.close()
            app.quit()
            driver.quit()
        except: pass

    print("\n✅ AUTOMAÇÃO N1 FINALIZADA!")

main()

Instalando python-dateutil...
✓ Dependências checadas.
🚀 Iniciando automação N1...
✅ Chrome inicializado.

=== VERIFICANDO LOGIN ===
✅ Sessão já ativa.
📊 Abrindo planilha...
📊 1 linha(s) a processar.


############################################################
📍 1/1
############################################################

🟩 N1 — LINHA 2847
   NF: 44 | Loja: 124 | Projeto: MISSOES HIGIENICAS
   Fornecedor: BENEL | Valor: 59.650,00
   Filial: 124 | Bandeira: PM | Ordem: 606114
   Vencimento: 19/05/2026
   PDF: ✅
⏳ 1. Serviços financeiros → Regularização de Notas Fiscais
⏳ 2. Regularização NF → Lançamento Reg. NF de Serviço
⏳ 3. Setor de Aprovação → Obras
⏳ 4. Centro de Custo → 124
⏳ 5. Forma de Pagamento → Crédito em Conta
⏳ 6. Ordem → 606114
⏳ 7. Valor → 59.650,00
⏳ 8. Data de Vencimento → 19/05/2026
⏳ 10. Bandeira → PM
⏳ 11. Fornecedor → BENEL
⏳ 12. Material → Material Ativo
⏳ 13. Cód Material 20103 → incluído na descrição
⏳ 14. Local de Atendimento → Matriz
⏳ 15. Assunto
⏳ 16. 